# Reinforcement Learning - Lab 1: Understanding RL through TicTacToe
# J. Martinet
# Enhanced Educational Version


=============================================================================
WELCOME TO REINFORCEMENT LEARNING! 🎮
=============================================================================

Imagine you're teaching a puppy new tricks:
- The puppy tries different actions (sit, stay, roll over)
- When it does something good, you give it a treat (reward)
- Over time, the puppy learns which actions lead to treats

That's exactly what Reinforcement Learning is! But instead of a puppy, 
we're teaching a computer program to make smart decisions.

=============================================================================
THE BIG PICTURE: Key Concepts Explained 📚
=============================================================================

1. AGENT (The Learner) 🤖
   - This is our AI that needs to learn
   - Like a player in a game
   - Example: In TicTacToe, the agent is the X or O player

2. ENVIRONMENT (The World) 🌍
   - The world where the agent lives and acts
   - Example: The TicTacToe board and game rules

3. STATE (The Situation) 📸
   - A snapshot of what's happening right now
   - Example: The current arrangement of X's and O's on the board
   
4. ACTION (The Choice) 🎯
   - What the agent can do
   - Example: Placing your X or O in an empty square

5. REWARD (The Feedback) 🏆
   - A number telling the agent if it did well or poorly
   - Example: +1 for winning, -1 for losing, 0 for a draw

6. POLICY (The Strategy) 🧠
   - The agent's rule for choosing actions
   - Example: "Always try to get three in a row, and block opponent"

7. VALUE FUNCTION (The Predictor) 🔮
   - Estimates how good each state is
   - Example: "If I have two X's in a row, I'm likely to win!"

=============================================================================
THE GOAL 🎯
=============================================================================

Make the agent learn the BEST POLICY - the strategy that wins most often!

The agent will:
1. Play many games
2. Learn from wins and losses
3. Update its value function (predictions)
4. Get better over time

=============================================================================
TEMPORAL DIFFERENCE (TD) LEARNING 🕐
=============================================================================

This is our learning method! Think of it like this:

You're walking to school and estimate: "I'll arrive in 20 minutes"
After 5 minutes, you realize: "Actually, I'll arrive in 18 minutes"
You UPDATE your estimate based on new information!

TD Formula: V(state) = V(state) + α × [V(next_state) - V(state)]
                                      ↑
                                This is the "error" - 
                                the difference between what you 
                                thought and what you now think

α (alpha) = Learning rate (how much to update, usually 0.1 to 0.5)

=============================================================================
REAL WORLD APPLICATIONS 🌟
=============================================================================

Where is this used?

1. GAME AI 🎮
   - Chess computers (Deep Blue, AlphaGo)
   - Video game characters that adapt to your play style

2. ROBOTS 🤖
   - Self-driving cars learning to navigate
   - Robot arms learning to grab objects

3. FINANCE 💰
   - Trading algorithms that learn market patterns
   - Investment portfolio optimization

4. HEALTHCARE 🏥
   - Personalized treatment recommendations
   - Drug discovery optimization

5. RECOMMENDATIONS 📱
   - Netflix/YouTube suggesting videos you'll like
   - Shopping recommendations on Amazon

In [1]:

# ============================================================================
# PART 1: Installation Check
# ============================================================================

print("="*70)
print("PART 1: Checking Installation")
print("="*70)

try:
    import gymnasium as gym
    print("✓ Gymnasium is already installed!")
except:
    print("Installing Gymnasium...")
    import subprocess
    subprocess.check_call(['pip3', 'install', 'gymnasium'])
    import gymnasium as gym
    print("✓ Gymnasium installed successfully!")

import numpy as np
import random
from typing import Tuple, Optional

print("\n✓ All imports successful! Let's start learning!\n")


PART 1: Checking Installation
✓ Gymnasium is already installed!

✓ All imports successful! Let's start learning!



In [2]:


# ============================================================================
# PART 2: Building the TicTacToe Environment
# ============================================================================

print("="*70)
print("PART 2: Creating the TicTacToe Environment")
print("="*70)

class TicTacToe:
    """
    This is our ENVIRONMENT - the world where our agent plays.
    
    The board is represented as a 3x3 grid:
    - 0 = empty square
    - 1 = player 1 (X)
    - 2 = player 2 (O)
    """
    
    def __init__(self):
        """Initialize a new game"""
        # STATE: Empty 3x3 board (all zeros)
        self.board = np.zeros((3, 3), dtype=int)
        self.current_player = 1  # Player 1 (X) starts
        
    def reset(self):
        """Start a new game (reset the STATE)"""
        self.board = np.zeros((3, 3), dtype=int)
        self.current_player = 1
        return self.get_state()
    
    def get_state(self):
        """
        Get the current STATE as a tuple (so it can be used as a dictionary key)
        This is like taking a photo of the current board
        """
        return tuple(self.board.flatten())
    
    def get_valid_actions(self):
        """
        Get all possible ACTIONS (empty squares where we can play)
        Returns list of (row, col) positions
        """
        actions = []
        for i in range(3):
            for j in range(3):
                if self.board[i, j] == 0:  # If square is empty
                    actions.append((i, j))
        return actions
    
    def take_action(self, action: Tuple[int, int]):
        """
        Perform an ACTION - place current player's mark on the board
        
        Args:
            action: (row, col) tuple indicating where to place the mark
            
        Returns:
            new_state: The new board state after the action
            reward: The REWARD (1 for win, -1 for loss, 0 for continue/draw)
            done: Whether the game is over
        """
        row, col = action
        
        # Place the mark
        self.board[row, col] = self.current_player
        
        # Check if current player won
        if self.check_winner(self.current_player):
            reward = 1  # Win! 
            done = True
        # Check if board is full (draw)
        elif len(self.get_valid_actions()) == 0:
            reward = 0  # Draw (tie game)
            done = True
        else:
            reward = 0  # Game continues
            done = False
        
        # Switch to other player
        self.current_player = 3 - self.current_player  # Switches 1→2 or 2→1
        
        return self.get_state(), reward, done
    
    def check_winner(self, player):
        """
        Check if the specified player has won
        Checks rows, columns, and diagonals
        """
        # Check rows
        for i in range(3):
            if all(self.board[i, :] == player):
                return True
        
        # Check columns
        for j in range(3):
            if all(self.board[:, j] == player):
                return True
        
        # Check diagonals
        if all(self.board.diagonal() == player):
            return True
        if all(np.fliplr(self.board).diagonal() == player):
            return True
        
        return False
    
    def display(self):
        """Display the board in a human-readable way"""
        symbols = {0: '.', 1: 'X', 2: 'O'}
        print("\n  0 1 2")
        for i in range(3):
            print(f"{i}", end=" ")
            for j in range(3):
                print(symbols[self.board[i, j]], end=" ")
            print()
        print()

# Test our environment!
print("\nLet's test our TicTacToe environment:")
game = TicTacToe()
game.display()
print("Valid actions:", game.get_valid_actions())
print("✓ Environment created successfully!\n")

PART 2: Creating the TicTacToe Environment

Let's test our TicTacToe environment:

  0 1 2
0 . . . 
1 . . . 
2 . . . 

Valid actions: [(0, 0), (0, 1), (0, 2), (1, 0), (1, 1), (1, 2), (2, 0), (2, 1), (2, 2)]
✓ Environment created successfully!



In [3]:


# ============================================================================
# PART 3: Creating the RL Agent with TD Learning
# ============================================================================

print("="*70)
print("PART 3: Creating our Learning Agent")
print("="*70)

class TDAgent:
    """
    This is our AGENT - the learner!
    
    It uses Temporal Difference (TD) learning to improve over time.
    The agent maintains a VALUE FUNCTION that estimates how good each state is.
    """
    
    def __init__(self, player_num, alpha=0.3, epsilon=0.2):
        """
        Initialize the agent
        
        Args:
            player_num: Which player (1 or 2)
            alpha: Learning rate (how fast to learn, 0-1)
            epsilon: Exploration rate (how often to try random moves, 0-1)
        """
        self.player_num = player_num
        self.alpha = alpha  # Learning rate
        self.epsilon = epsilon  # Exploration rate
        
        # VALUE FUNCTION: Dictionary mapping states to estimated win probability
        # Initially, all states are estimated at 0.5 (50% chance of winning)
        self.values = {}
        
        # Remember states visited in current game for learning
        self.state_history = []
        
    def get_value(self, state):
        """
        Get the estimated value of a state
        If we haven't seen this state before, assume neutral (0.5)
        """
        if state not in self.values:
            self.values[state] = 0.5  # Neutral starting estimate
        return self.values[state]
    
    def choose_action(self, game, training=True):
        """
        POLICY: How the agent chooses actions
        
        Uses ε-greedy policy:
        - With probability ε: explore (random action)
        - With probability 1-ε: exploit (best known action)
        
        This balance between exploration and exploitation is crucial!
        """
        valid_actions = game.get_valid_actions()
        
        # EXPLORATION: Sometimes try random moves to discover new strategies
        if training and random.random() < self.epsilon:
            return random.choice(valid_actions)
        
        # EXPLOITATION: Choose the action leading to best estimated state
        best_value = -float('inf')
        best_action = None
        
        for action in valid_actions:
            # Simulate taking this action
            game_copy = TicTacToe()
            game_copy.board = game.board.copy()
            game_copy.current_player = game.current_player
            
            next_state, _, _ = game_copy.take_action(action)
            value = self.get_value(next_state)
            
            if value > best_value:
                best_value = value
                best_action = action
        
        return best_action
    
    def learn(self, reward):
        """
        TEMPORAL DIFFERENCE LEARNING! 🎓
        
        This is where the magic happens!
        We update our value estimates based on the game outcome.
        
        TD Update Rule:
        V(s_t) ← V(s_t) + α × [V(s_{t+1}) - V(s_t)]
        
        Think of it as:
        "My new estimate = old estimate + learning_rate × prediction_error"
        """
        # Work backwards through the states we visited
        for i in range(len(self.state_history) - 1):
            current_state = self.state_history[i]
            next_state = self.state_history[i + 1]
            
            # Current estimate
            current_value = self.get_value(current_state)
            # Next estimate (what we learned from the next state)
            next_value = self.get_value(next_state)
            
            # TD UPDATE: Move our estimate towards the next state's estimate
            # This propagates the final reward backwards through the game
            td_error = next_value - current_value
            self.values[current_state] = current_value + self.alpha * td_error
        
        # For the final state, update towards the actual reward
        if self.state_history:
            final_state = self.state_history[-1]
            final_value = self.get_value(final_state)
            self.values[final_state] = final_value + self.alpha * (reward - final_value)
        
        # Clear history for next game
        self.state_history = []
    
    def add_state(self, state):
        """Remember this state for learning later"""
        self.state_history.append(state)

print("✓ Agent created with TD learning capabilities!\n")



PART 3: Creating our Learning Agent
✓ Agent created with TD learning capabilities!



In [6]:
# ============================================================================
# PART 4: Training the Agent
# ============================================================================

print("="*70)
print("PART 4: Training our Agent")
print("="*70)

def play_game(agent1, agent2, training=True, display=False):
    """
    Play one complete game between two agents
    
    Returns:
        winner: 1, 2, or 0 (draw)
    """
    game = TicTacToe()
    agents = {1: agent1, 2: agent2}
    
    if display:
        print("\n--- New Game ---")
        game.display()
    
    while True:
        current_agent = agents[game.current_player]
        
        # Remember current state for learning
        if training:
            current_agent.add_state(game.get_state())
        
        # Choose and take action
        action = current_agent.choose_action(game, training)
        new_state, reward, done = game.take_action(action)
        
        if display:
            print(f"Player y{3 - game.current_player} plays at {action}")
            game.display()
        
        if done:
            # Game over! Distribute rewards and learn
            if reward == 1:  # Current player won
                winner = 3 - game.current_player  # (it switched already)
                if training:
                    agents[winner].learn(1)  # Winner gets +1 reward
                    agents[3 - winner].learn(-1)  # Loser gets -1 reward
                return winner
            else:  # Draw
                if training:
                    agent1.learn(0)
                    agent2.learn(0)
                return 0

# Create two learning agents
print("\nCreating two learning agents...")
agent_x = TDAgent(player_num=1, alpha=0.3, epsilon=0.2)
agent_o = TDAgent(player_num=2, alpha=0.3, epsilon=0.2)

# Train for many games!
print("\n Training agents by playing games...")
print("This is where the agents LEARN from experience!\n")

num_training_games = 5000
wins = {1: 0, 2: 0, 0: 0}  # Track wins, losses, draws

for episode in range(num_training_games):
    winner = play_game(agent_x, agent_o, training=True, display=False)
    wins[winner] += 1
    
    # Print progress every 1000 games
    if (episode + 1) % 1000 == 0:
        total = episode + 1
        print(f"Episode {total:5d} | X wins: {wins[1]:4d} ({wins[1]/total*100:.1f}%) | "
              f"O wins: {wins[2]:4d} ({wins[2]/total*100:.1f}%) | "
              f"Draws: {wins[0]:4d} ({wins[0]/total*100:.1f}%)")

print(f"\n✓ Training complete! Agents learned from {num_training_games} games.")
print(f"✓ Agent X learned {len(agent_x.values)} different board positions")
print(f"✓ Agent O learned {len(agent_o.values)} different board positions")


PART 4: Training our Agent

Creating two learning agents...

 Training agents by playing games...
This is where the agents LEARN from experience!

Episode  1000 | X wins:  694 (69.4%) | O wins:  207 (20.7%) | Draws:   99 (9.9%)
Episode  2000 | X wins: 1385 (69.2%) | O wins:  419 (20.9%) | Draws:  196 (9.8%)
Episode  3000 | X wins: 2114 (70.5%) | O wins:  599 (20.0%) | Draws:  287 (9.6%)
Episode  4000 | X wins: 2846 (71.2%) | O wins:  779 (19.5%) | Draws:  375 (9.4%)
Episode  5000 | X wins: 3592 (71.8%) | O wins:  936 (18.7%) | Draws:  472 (9.4%)

✓ Training complete! Agents learned from 5000 games.
✓ Agent X learned 1841 different board positions
✓ Agent O learned 1777 different board positions


In [7]:

# ============================================================================
# PART 5: Testing the Trained Agent
# ============================================================================

print("\n" + "="*70)
print("PART 5: Testing the Trained Agent")
print("="*70)

print("\nNow let's watch our trained agent play!")
print("Notice how it makes smart moves because it LEARNED from experience.\n")

# Play a few demonstration games
for i in range(3):
    print(f"\n{'='*40}")
    print(f"Demonstration Game {i+1}")
    print(f"{'='*40}")
    winner = play_game(agent_x, agent_o, training=False, display=True)
    if winner == 1:
        print(" Player X (Agent 1) wins!")
    elif winner == 2:
        print(" Player O (Agent 2) wins!")
    else:
        print(" It's a draw!")



PART 5: Testing the Trained Agent

Now let's watch our trained agent play!
Notice how it makes smart moves because it LEARNED from experience.


Demonstration Game 1

--- New Game ---

  0 1 2
0 . . . 
1 . . . 
2 . . . 

Player y1 plays at (0, 0)

  0 1 2
0 X . . 
1 . . . 
2 . . . 

Player y2 plays at (0, 1)

  0 1 2
0 X O . 
1 . . . 
2 . . . 

Player y1 plays at (0, 2)

  0 1 2
0 X O X 
1 . . . 
2 . . . 

Player y2 plays at (1, 0)

  0 1 2
0 X O X 
1 O . . 
2 . . . 

Player y1 plays at (1, 1)

  0 1 2
0 X O X 
1 O X . 
2 . . . 

Player y2 plays at (1, 2)

  0 1 2
0 X O X 
1 O X O 
2 . . . 

Player y1 plays at (2, 0)

  0 1 2
0 X O X 
1 O X O 
2 X . . 

 Player X (Agent 1) wins!

Demonstration Game 2

--- New Game ---

  0 1 2
0 . . . 
1 . . . 
2 . . . 

Player y1 plays at (0, 0)

  0 1 2
0 X . . 
1 . . . 
2 . . . 

Player y2 plays at (0, 1)

  0 1 2
0 X O . 
1 . . . 
2 . . . 

Player y1 plays at (0, 2)

  0 1 2
0 X O X 
1 . . . 
2 . . . 

Player y2 plays at (1, 0)

  0 1 2
0 X O X 
1

In [10]:

# ============================================================================
# PART 6: Interactive Play (Optional)
# ============================================================================

print("\n" + "="*70)
print("PART 6: Play Against the Trained Agent!")
print("="*70)

def play_with_human():
    """Let a human play against the trained agent"""
    game = TicTacToe()
    print("\nYou are O, the agent is X")
    print("Enter your move as: row col (e.g., '0 1' for top middle)")
    game.display()
    
    while True:
        # Agent's turn (X)
        agent_x.add_state(game.get_state())
        action = agent_x.choose_action(game, training=False)
        print(f"Agent plays at {action}")
        new_state, reward, done = game.take_action(action)
        game.display()
        
        if done:
            if reward == 1:
                print("😔 Agent wins! Better luck next time!")
            else:
                print("🤝 It's a draw!")
            return
        
        # Human's turn (O)
        while True:
            try:
                move = input("Your move (row col): ")
                row, col = map(int, move.split())
                if (row, col) in game.get_valid_actions():
                    break
                else:
                    print("Invalid move! Square is occupied or out of bounds.")
            except:
                print("Invalid input! Enter as: row col (e.g., '0 1')")
        
        new_state, reward, done = game.take_action((row, col))
        game.display()
        
        if done:
            if reward == 1:
                print("🎉 You win! Great job!")
            else:
                print("🤝 It's a draw!")
            return

print("\nUncomment the line below to play against the agent:")
play_with_human()
print("# play_with_human()")



PART 6: Play Against the Trained Agent!

Uncomment the line below to play against the agent:

You are O, the agent is X
Enter your move as: row col (e.g., '0 1' for top middle)

  0 1 2
0 . . . 
1 . . . 
2 . . . 

Agent plays at (0, 0)

  0 1 2
0 X . . 
1 . . . 
2 . . . 

Invalid input! Enter as: row col (e.g., '0 1')

  0 1 2
0 X . . 
1 . O . 
2 . . . 

Agent plays at (0, 1)

  0 1 2
0 X X . 
1 . O . 
2 . . . 

Invalid input! Enter as: row col (e.g., '0 1')
Invalid input! Enter as: row col (e.g., '0 1')
Invalid input! Enter as: row col (e.g., '0 1')
Invalid input! Enter as: row col (e.g., '0 1')
Invalid move! Square is occupied or out of bounds.
Invalid input! Enter as: row col (e.g., '0 1')
Invalid input! Enter as: row col (e.g., '0 1')
Invalid move! Square is occupied or out of bounds.

  0 1 2
0 X X . 
1 . O O 
2 . . . 

Agent plays at (0, 2)

  0 1 2
0 X X X 
1 . O O 
2 . . . 

😔 Agent wins! Better luck next time!
# play_with_human()


In [13]:

# ============================================================================
# SUMMARY AND CONCLUSION
# ============================================================================

print("\n" + "="*70)
print("🎓 WHAT WE LEARNED")
print("="*70)

summary = """
1. REINFORCEMENT LEARNING BASICS:
   - Agent learns by interacting with an environment
   - Gets rewards for good actions, penalties for bad ones
   - Goal: Learn the best policy (strategy) to maximize rewards

2. TEMPORAL DIFFERENCE (TD) LEARNING:
   - Updates value estimates based on experience
   - Formula: V(s) ← V(s) + α × [V(s') - V(s)]
   - Learns by comparing predictions at different time steps

3. KEY CONCEPTS WE IMPLEMENTED:
   ✓ Environment: TicTacToe game board and rules
   ✓ State: Current board configuration
   ✓ Action: Placing X or O in an empty square
   ✓ Reward: +1 for win, -1 for loss, 0 for draw
   ✓ Policy: ε-greedy (explore vs exploit)
   ✓ Value Function: Estimated win probability for each state

4. EXPLORATION vs EXPLOITATION:
   - Exploration: Try new things to learn
   - Exploitation: Use what you know to win
   - Balance is crucial for good learning!

5. WHAT WE ACHIEVED:
   - Trained an agent to play TicTacToe
   - Agent learned from {} games
   - No explicit rules taught - it learned by playing!
   
6. REAL WORLD IMPACT:
   This same technique (with improvements) is used in:
   - AlphaGo (beat world champion at Go)
   - Self-driving cars
   - Robot control
   - Game AI
   - And much more!

The beautiful thing about RL: The agent discovers strategies on its own,
sometimes finding solutions humans never thought of! 
""".format(num_training_games)

print(summary)

print("\n" + "="*70)
print(" LAB COMPLETE! You now understand Reinforcement Learning!")
print("="*70)


🎓 WHAT WE LEARNED

1. REINFORCEMENT LEARNING BASICS:
   - Agent learns by interacting with an environment
   - Gets rewards for good actions, penalties for bad ones
   - Goal: Learn the best policy (strategy) to maximize rewards

2. TEMPORAL DIFFERENCE (TD) LEARNING:
   - Updates value estimates based on experience
   - Formula: V(s) ← V(s) + α × [V(s') - V(s)]
   - Learns by comparing predictions at different time steps

3. KEY CONCEPTS WE IMPLEMENTED:
   ✓ Environment: TicTacToe game board and rules
   ✓ State: Current board configuration
   ✓ Action: Placing X or O in an empty square
   ✓ Reward: +1 for win, -1 for loss, 0 for draw
   ✓ Policy: ε-greedy (explore vs exploit)
   ✓ Value Function: Estimated win probability for each state

4. EXPLORATION vs EXPLOITATION:
   - Exploration: Try new things to learn
   - Exploitation: Use what you know to win
   - Balance is crucial for good learning!

5. WHAT WE ACHIEVED:
   - Trained an agent to play TicTacToe
   - Agent learned from 500